In [16]:
%pip install numpy pandas 

Note: you may need to restart the kernel to use updated packages.


In [22]:
import pandas as pd
import csv
from collections import defaultdict
from collections import Counter

orders_path = '../dataset/Transaction/orders.csv'
product_path = '../dataset/Master/products.csv'
returns_path = '../dataset/Transaction/returns.csv'
web_traffic_path = '../dataset/Operational/web_traffic.csv'
order_items_path = '../dataset/Transaction/order_items.csv'
customers_path = '../dataset/Master/customers.csv'
geography_path = '../dataset/Master/geography.csv'
payments_path = '../dataset/Transaction/payments.csv'

Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

A) 30 ngày

B) 90 ngày

C) 180 ngày

D) 365 ngày

In [23]:
df = pd.read_csv(orders_path)

# 1. Lọc chỉ lấy những đơn hàng 'delivered'
df = df[df['order_status'] == 'delivered']

# 2. Chuyển đổi sang datetime
df['order_date'] = pd.to_datetime(df['order_date'])

# 3. Loại bỏ các đơn hàng mua cùng ngày của cùng 1 khách hàng (nếu bạn muốn tính gap giữa các NGÀY mua khác nhau)
# Bỏ dòng này nếu bạn vẫn muốn coi 2 order trên cùng 1 ngày có gap = 0
df = df.drop_duplicates(subset=['customer_id', 'order_date'])

# 4. Sắp xếp theo khách hàng và ngày mua
df = df.sort_values(by=['customer_id', 'order_date'])

# 5. Lọc những khách hàng có từ 2 đơn hàng (delivered) trở lên
counts = df['customer_id'].value_counts()
khach_hang_nhieu_don = counts[counts >= 2].index
df = df[df['customer_id'].isin(khach_hang_nhieu_don)]

# 6. Tính khoảng cách số ngày giữa 2 lần mua liên tiếp cho từng khách hàng
df['gap'] = df.groupby('customer_id')['order_date'].diff().dt.days

# 7. Lấy trung vị
median_gap = df['gap'].median()
mean_gap = df['gap'].mean()

print(f"Trung vị (Median) số ngày giữa 2 lần mua liên tiếp (tính riêng status delivered): {median_gap} ngày")
print(f"Trung bình (Mean) số ngày giữa 2 lần mua liên tiếp (tính riêng status delivered): {mean_gap:.2f} ngày")


Trung vị (Median) số ngày giữa 2 lần mua liên tiếp (tính riêng status delivered): 178.0 ngày
Trung bình (Mean) số ngày giữa 2 lần mua liên tiếp (tính riêng status delivered): 327.58 ngày


Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price − cogs)/price?

A) Premium

B) Performance

C) Activewear

**D) Standard**

In [24]:
# Đọc dữ liệu sản phẩm
df_products = pd.read_csv(product_path)

# Tính margin gộp cho từng sản phẩm: (price - cogs) / priceẩm: (price - cogs) / price
df_products['gross_margin'] = (df_products['price'] - df_products['cogs']) / df_products['price']

# Tính trung bình theo từng phân khúc (segment) và sắp xếp giảm dần
margin_by_segment = df_products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)

print(margin_by_segment)


segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64


Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

A) defective

**B) wrong_size**

C) changed_mind

D) not_as_described

In [25]:
df_returns = pd.read_csv(returns_path)
df_products = pd.read_csv(product_path)

# Gộp dữ liệu returns với bảng sản phẩm (products) qua product_id (products) qua product_id
df_joined = pd.merge(df_returns, df_products, on='product_id')

# Lọc chỉ những sản phẩm thuộc danh mục "Streetwear"
df_streetwear = df_joined[df_joined['category'] == 'Streetwear']

# Đếm số lượng theo nhóm lý do trả hàng
reason_counts = df_streetwear['return_reason'].value_counts()
print(reason_counts)


return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

A) organic_search

B) paid_search

**C) email_campaign**

D) social_media

In [26]:
df = pd.read_csv(web_traffic_path)
mean_bounce_rate = df.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print(mean_bounce_rate)


traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64


Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

A) 12%

B) 25%

**C) 39%**

D) 54%

In [27]:
df = pd.read_csv(order_items_path, low_memory=False)
pct = df['promo_id'].notna().mean()*100
print(f'rows = {len(df)}')
print(f'promo_not_null = {df["promo_id"].notna().sum()}')
print(f'pct = {pct:.4f}')

rows = 714669
promo_not_null = 276316
pct = 38.6635


Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

**A) 55+**

B) 25–34

C) 35–44

D) 45–54

In [28]:
customers={}

with open(customers_path, newline='', encoding='utf-8') as f:
    r=csv.DictReader(f)
    for row in r:
        cid=row['customer_id']
        age=row.get('age_group')
        if age is not None:
            age=age.strip()
        customers[cid]=age

orders_per_customer=defaultdict(int)

with open(orders_path, newline='', encoding='utf-8') as f:
    r=csv.DictReader(f)
    for row in r:
        cid=row['customer_id']
        orders_per_customer[cid]+=1

sum_orders=defaultdict(int)
count_customers=defaultdict(int)

for cid, age in customers.items():
    if age is None or age=='':
        continue
    if cid in orders_per_customer:
        sum_orders[age]+=orders_per_customer[cid]
        count_customers[age]+=1

avg_by_age={age: sum_orders[age]/count_customers[age] for age in count_customers}
for age, avg in sorted(avg_by_age.items(), key=lambda x: x[1], reverse=True):
    print(age, avg, 'customers=', count_customers[age], 'orders=', sum_orders[age])

55+ 7.268731268731269 customers= 10010 orders= 72760
45-54 7.2202640609550395 customers= 17193 orders= 124138
35-44 7.206158531427121 customers= 23642 orders= 170368
25-34 7.112230430564884 customers= 26802 orders= 190622
18-24 7.068576871180253 customers= 12599 orders= 89057


Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?

A) West

B) Central

**C) East**

D) Cả ba vùng có doanh thu xấp xỉ bằng nhau

In [29]:
geo={}
with open(geography_path, newline='', encoding='utf-8') as f:
    for r in csv.DictReader(f):
        geo[r['zip']]=r['region']

order_region={}
order_status={}
with open(orders_path, newline='', encoding='utf-8') as f:
    for r in csv.DictReader(f):
        oid=r['order_id']
        z=r['zip']
        order_region[oid]=geo.get(z)
        order_status[oid]=r.get('order_status','')

sum_all=defaultdict(float)
sum_delivered=defaultdict(float)
with open(order_items_path, newline='', encoding='utf-8') as f:
    for r in csv.DictReader(f):
        oid=r['order_id']
        region=order_region.get(oid)
        if not region:
            continue
        qty=float(r['quantity']) if r['quantity'] else 0.0
        unit=float(r['unit_price']) if r['unit_price'] else 0.0
        disc=float(r['discount_amount']) if r['discount_amount'] else 0.0
        rev=qty*unit-disc
        sum_all[region]+=rev
        if order_status.get(oid)=='delivered':
            sum_delivered[region]+=rev

print('ALL:')
for k,v in sorted(sum_all.items(), key=lambda x:x[1], reverse=True):
    print(k, round(v,2))
print('\nDELIVERED:')
for k,v in sorted(sum_delivered.items(), key=lambda x:x[1], reverse=True):
    print(k, round(v,2))

# spread check
vals=[v for _,v in sorted(sum_all.items())]
if vals:
    print('\nall spread pct=', (max(vals)-min(vals))/max(vals)*100)
vals=[v for _,v in sorted(sum_delivered.items())]
if vals:
    print('delivered spread pct=', (max(vals)-min(vals))/max(vals)*100)

ALL:
East 7291150819.12
Central 4719491267.84
West 3670227178.47

DELIVERED:
East 5826256457.02
Central 3762669980.47
West 2929249519.71

all spread pct= 49.66189467861007
delivered spread pct= 49.72329932060243


Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức
thanh toán nào được sử dụng nhiều nhất?

A) credit_card

B) cod

C) paypal

D) bank_transfer

In [30]:
ancelled_ids = set()
with open(orders_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['order_status'] == 'cancelled':
            cancelled_ids.add(row['order_id'])

print(f'Total cancelled orders: {len(cancelled_ids)}')
payment_counts = Counter()
with open('../dataset/Transaction/payments.csv', mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['order_id'] in cancelled_ids:
            payment_counts[row['payment_method']] += 1

sorted_counts = sorted(payment_counts.items(), key=lambda x: x[1], reverse=True)
for method, count in sorted_counts:
    print(f'{method}: {count}')
if sorted_counts:
    print(f'Top method: {sorted_counts[0][0]}')

Total cancelled orders: 59462
credit_card: 28452
cod: 15468
paypal: 7817
apple_pay: 5190
bank_transfer: 2535
Top method: credit_card


Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao
nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join
với products theo product_id)?

A) S

B) M

C) L

D) XL


In [31]:
def get_counts(file_path):
    counts = Counter()
    with open(file_path, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            counts[row['product_id']] += 1
    return counts

product_size = {}
with open('../dataset/Master/products.csv', mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['size'] in {'S', 'M', 'L', 'XL'}:
            product_size[row['product_id']] = row['size']

order_counts = get_counts(order_items_path)
return_counts = get_counts(returns_path)

size_orders = Counter()
size_returns = Counter()

for pid, count in order_counts.items():
    if pid in product_size:
        size_orders[product_size[pid]] += count

for pid, count in return_counts.items():
    if pid in product_size:
        size_returns[product_size[pid]] += count

results = []
for s in ['S', 'M', 'L', 'XL']:
    o = size_orders[s]
    r = size_returns[s]
    rate = r/o if o > 0 else 0
    results.append({'Size': s, 'Orders': o, 'Returns': r, 'Rate': rate})

results.sort(key=lambda x: x['Rate'], reverse=True)
for res in results:
    print(f'Size {res['Size']}: Orders={res['Orders']}, Returns={res['Returns']}, Rate={res['Rate']:.4f}')

if results:
    print(f'Top size: {results[0]['Size']}')

Size S: Orders=172042, Returns=9723, Rate=0.0565
Size L: Orders=173174, Returns=9741, Rate=0.0562
Size M: Orders=176428, Returns=9820, Rate=0.0557
Size XL: Orders=193025, Returns=10655, Rate=0.0552
Top size: S


Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
mỗi đơn hàng cao nhất?

A) 1 kỳ (trả một lần)

B) 3 kỳ

C) 6 kỳ

D) 12 kỳ

In [32]:
import pandas as pd
df = pd.read_csv(payments_path)
avg_payments = df.groupby('installments')['payment_value'].mean()
print(avg_payments.sort_values(ascending=False))
print('Top plan:', avg_payments.idxmax())

installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64
Top plan: 6
